<a href="https://colab.research.google.com/github/cmauriciocrescente-upc/Dataset-en-Colab/blob/main/Actividad_14_10_26_Crescente_Mauricio_35564222_Parte2_y_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad Práctica Completa: Procesamiento y Ciencia de Datos
**Alumno:** Crescente Mauricio

# Parte 2
## Consignas de segunda parte
Se repite el pipeline completo de la Parte 1 (generación de un *dirty dataset* + 7 pasos + checklist) sobre un **segundo dataset sintético de otro dominio**: *abandono de clientes (churn) en una empresa de telecomunicaciones*. A diferencia de la Parte 1, el target depende realmente de las variables (antigüedad, tipo de contrato y cargo mensual), de modo que el modelo final tiene señal para aprender y las métricas son interpretables.

| # | Etapa | Qué se hace en este dataset |
|---|-------|------------------------------|
| 1 | Diagnóstico y Auditoría | Calidad (completitud, consistencia, validez, unicidad) y detección de 2 variables con *Target Leakage* |
| 2 | Limpieza Operativa | Duplicados, texto, moneda (`$`, `,`, "Sin cargo"), fechas, rangos de edad |
| 3 | Datos Faltantes | Test de mecanismos MCAR/MAR, descarte por umbral >50 %, imputación condicional |
| 4 | Outliers | IQR, Z-Score y Winsorización |
| 5 | Partición | `train_test_split(stratify=y)` |
| 6 | Transformación | `ColumnTransformer` (Yeo-Johnson, StandardScaler, OneHot, Ordinal) |
| 7 | Balanceo y Evaluación | SMOTE solo en train, Random Forest *cost-sensitive*, *moving threshold*, PR-AUC |

## 2.1 Generación del Dataset Sintético "Dirty Dataset" (Churn Telecom)
El script genera `dataset_churn_telecom_dirty.csv` con: duplicados exactos, fechas heterogéneas e imposibles (1900, 2035), edades fuera de rango (-5, 130), cargo mensual como texto (`$`, comas, "Sin cargo"), nulos MCAR (edad) y MAR (cargo según tipo de contrato), categorías con espacios y mayúsculas inconsistentes, una columna con >50 % de faltantes, outliers extremos en el consumo de datos, desbalanceo de clases (~7 %) y **dos** variables con *Target Leakage*: `dias_hasta_baja` y `motivo_baja`.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(7)
n_samples = 1200

# 1. Variables "reales" (antes de inyectar errores)
antiguedad = np.random.gamma(shape=2.0, scale=14.0, size=n_samples).round(0) + 1      # meses, asimétrica
contrato_limpio = np.random.choice(['Mensual', 'Anual', 'Bianual'], size=n_samples, p=[0.55, 0.30, 0.15])
plan_limpio = np.random.choice(['Básico', 'Estándar', 'Premium', 'Platino'], size=n_samples, p=[0.35, 0.35, 0.20, 0.10])
cargo_base = {'Básico': 25, 'Estándar': 45, 'Premium': 70, 'Platino': 95}
cargo = np.array([cargo_base[p] for p in plan_limpio]) * np.random.uniform(0.8, 1.3, size=n_samples)

# 2. Target con señal real (regresión logística latente): menos antigüedad, contrato mensual y cargo alto => más abandono
logit = (-2.4 - 0.035 * antiguedad
         + 1.1 * (contrato_limpio == 'Mensual') - 0.8 * (contrato_limpio == 'Bianual')
         + 0.015 * (cargo - 50))
prob = 1 / (1 + np.exp(-logit))
target = (np.random.rand(n_samples) < prob).astype(int)

# 3. Identificadores
ids = np.arange(10001, 10001 + n_samples)

# 4. Fechas con formatos heterogéneos e inconsistencias
fechas_base = ['2023-01-15', '15/02/2023', '2023.03.10', '10-Apr-2023', '1900-01-01', '2035-12-31', None]
fechas = np.random.choice(fechas_base, size=n_samples)

# 5. Edades con valores fuera de rango (-5, 130) y faltantes MCAR
edades = np.random.normal(loc=42, scale=13, size=n_samples)
edades[10:20] = -5
edades[21:30] = 130
edades[31:45] = np.nan
edades[np.random.rand(n_samples) < 0.03] = np.nan      # MCAR: faltan al azar, sin relación con nada

# 6. Cargo mensual como texto con símbolos y errores
cargo_str = np.array([f"${x:,.2f}" for x in cargo], dtype=object)
cargo_str[60:80] = "Sin cargo"
# MAR: el cargo falta más seguido en clientes de contrato Mensual (depende de una variable OBSERVADA)
mask_mar = (contrato_limpio == 'Mensual') & (np.random.rand(n_samples) < 0.12)
cargo_str[mask_mar] = np.nan

# 7. Consumo de datos con asimetría positiva y outliers extremos
consumo_gb = np.random.lognormal(mean=3.0, sigma=0.8, size=n_samples)
consumo_gb[100:105] = 5000.0

# 8. Tipo de contrato con espacios y mayúsculas/minúsculas inconsistentes
variantes = {'Mensual': [' Mensual ', 'mensual', 'MENSUAL', 'Mensual'],
             'Anual': ['Anual', 'ANUAL ', 'anual'],
             'Bianual': ['Bianual ', 'bianual', 'BIANUAL']}
tipo_contrato = np.array([np.random.choice(variantes[c]) for c in contrato_limpio])

# 9. Columna con >50 % de faltantes
notas_soporte = [np.nan] * n_samples
for i in range(0, 100):
    notas_soporte[i] = "Reclamo por facturación"

# 10. Target Leakage (variables posteriores al evento)
dias_hasta_baja = np.where(target == 1, np.random.randint(5, 90, size=n_samples), 0)
motivo_baja = np.where(target == 1,
                       np.random.choice(['Precio', 'Competencia', 'Servicio'], size=n_samples),
                       'No aplica')

df_sintetico = pd.DataFrame({
    'id_cliente': ids,
    'fecha_contrato': fechas,
    'edad': edades,
    'antiguedad_meses': antiguedad,
    'cargo_mensual_str': cargo_str,
    'consumo_datos_gb': consumo_gb,
    'tipo_contrato': tipo_contrato,
    'nivel_plan': plan_limpio,
    'notas_soporte': notas_soporte,
    'dias_hasta_baja': dias_hasta_baja,     # Fuga de datos
    'motivo_baja': motivo_baja,             # Fuga de datos
    'abandono': target
})

# Inyección deliberada de duplicados exactos
df_sintetico = pd.concat([df_sintetico, df_sintetico.iloc[0:25]], ignore_index=True)
df_sintetico.to_csv('dataset_churn_telecom_dirty.csv', index=False)
print("Dataset sintético 'dataset_churn_telecom_dirty.csv' generado exitosamente.")

## 2.2 Guía Paso a Paso
### Paso 1: Exploración Inicial, Diagnóstico de Calidad y Auditoría Previa
**Objetivo:** inspeccionar la estructura, diagnosticar las cuatro dimensiones de calidad (Completitud, Consistencia, Validez, Unicidad) y auditar fuentes de *Target Leakage*.

- Carga con `.head()`, `.info()`, `.describe()`.
- Proporción de clases con `.value_counts(normalize=True)`.
- **Auditoría de Target Leakage:** una variable es sospechosa si (a) se registra *después* del evento o (b) separa el target de forma casi perfecta. Se verifica con el negocio **y** con los datos.

In [ ]:
import pandas as pd
import numpy as np

# Carga e inspección básica
df_raw = pd.read_csv('dataset_churn_telecom_dirty.csv')
print("--- Diagnóstico Estructural Inicial ---")
print(df_raw.head())
print(df_raw.info())
print(df_raw.describe())
print("\n--- Distribución de la Variable Objetivo ---")
print(df_raw['abandono'].value_counts(normalize=True))

In [ ]:
# --- Diagnóstico por dimensión de calidad ---
# 1) COMPLETITUD: % de datos presentes por columna
completitud = (df_raw.notna().mean() * 100).round(1).rename('completitud_%')
print("COMPLETITUD\n", completitud.sort_values(), "\n")

# 2) CONSISTENCIA: mismas categorías escritas distinto
print("CONSISTENCIA - tipo_contrato (valores crudos):")
print(df_raw['tipo_contrato'].map(repr).value_counts(), "\n")

# 3) VALIDEZ: valores imposibles según reglas de negocio
fechas_parseadas = pd.to_datetime(df_raw['fecha_contrato'], format='mixed', errors='coerce')
print("VALIDEZ")
print("  Edades fuera de [18, 100]:", ((df_raw['edad'] < 18) | (df_raw['edad'] > 100)).sum())
print("  Fechas anteriores a 2000 o futuras:", ((fechas_parseadas < '2000-01-01') | (fechas_parseadas > pd.Timestamp.today())).sum())
print("  Cargos 'Sin cargo' (texto no numérico):", (df_raw['cargo_mensual_str'] == 'Sin cargo').sum(), "\n")

# 4) UNICIDAD: filas completamente duplicadas
print("UNICIDAD")
print("  Filas duplicadas exactas:", df_raw.duplicated().sum())
print("  id_cliente repetidos:", df_raw['id_cliente'].duplicated().sum())

In [ ]:
# --- Auditoría de Target Leakage ---
# Si una variable separa el target casi perfectamente y solo existe tras el evento, es fuga.
print(df_raw.groupby('abandono')['dias_hasta_baja'].describe(), "\n")
print(pd.crosstab(df_raw['motivo_baja'], df_raw['abandono']))

# Eliminación preventiva de Target Leakage
df_clean = df_raw.drop(columns=['dias_hasta_baja', 'motivo_baja']).copy()
print("\nColumnas restantes:", list(df_clean.columns))

**Conclusión del Paso 1:** `dias_hasta_baja` es 0 para todos los clientes que no abandonaron y > 0 para todos los que sí; `motivo_baja` vale "No aplica" exactamente cuando no hubo abandono. Ambas se conocen únicamente *después* del evento que se quiere predecir, por lo que se eliminan del espacio de variables independientes.

### Paso 2: Limpieza Operativa, Deduplicación y Estandarización
**Objetivo:** remover duplicados, corregir tipos (casting), estandarizar cadenas y filtrar errores lógicos de negocio.

1. **Deduplicación:** `drop_duplicates(keep='first')`.
2. **Normalización categórica:** `.str.strip().str.title()` en `tipo_contrato`.
3. **Casting monetario:** quitar `$` y `,`; "Sin cargo" → 0; `pd.to_numeric(errors='coerce')`.
4. **Fechas:** `pd.to_datetime(..., format='mixed', errors='coerce')` y descarte de fechas imposibles (antes de 2000 o futuras).
5. **Rangos:** edades fuera de [18, 100] → `np.nan` con `np.where()`.

In [ ]:
# 1. Deduplicación
antes = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Filas: {antes} -> {len(df_clean)} (duplicados eliminados: {antes - len(df_clean)})")

# 2. Normalización de texto
df_clean['tipo_contrato'] = df_clean['tipo_contrato'].astype(str).str.strip().str.title()
print(df_clean['tipo_contrato'].value_counts())

# 3. Limpieza y casting de valores monetarios
df_clean['cargo_mensual'] = (
    df_clean['cargo_mensual_str']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('Sin cargo', '0', regex=False)
)
df_clean['cargo_mensual'] = pd.to_numeric(df_clean['cargo_mensual'], errors='coerce')
df_clean = df_clean.drop(columns=['cargo_mensual_str'])

# 4. Homogeneización de fechas + descarte de fechas imposibles
df_clean['fecha_contrato'] = pd.to_datetime(df_clean['fecha_contrato'], format='mixed', errors='coerce')
fecha_invalida = (df_clean['fecha_contrato'] < '2000-01-01') | (df_clean['fecha_contrato'] > pd.Timestamp.today())
df_clean['fecha_contrato'] = df_clean['fecha_contrato'].mask(fecha_invalida)

# 5. Corrección de rango de edad
df_clean['edad'] = np.where(
    (df_clean['edad'] < 18) | (df_clean['edad'] > 100),
    np.nan,
    df_clean['edad']
)
print(df_clean.dtypes)

### Paso 3: Visualización y Tratamiento de Datos Faltantes
**Objetivo:** evaluar los mecanismos de faltantes y aplicar eliminación por umbral e imputación condicional.

- **MCAR** (*Missing Completely At Random*): la probabilidad de faltar no depende de ningún dato.
- **MAR** (*Missing At Random*): depende de otras variables **observadas**. Se puede detectar: se compara la tasa de nulos entre grupos con un test chi-cuadrado.
- **MNAR** (*Missing Not At Random*): depende del propio valor faltante (ej.: quien gana mucho no declara ingresos). **No se puede demostrar con los datos**; se argumenta con conocimiento del negocio.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Visualización de matriz de nulos
plt.figure(figsize=(9, 4))
sns.heatmap(df_clean.isnull(), cbar=False, cmap='viridis')
plt.title('Mapa de Calor de Valores Nulos')
plt.show()

print("% de nulos por columna:")
print((df_clean.isnull().mean() * 100).round(1).sort_values(ascending=False))

In [ ]:
# Identificación del mecanismo: ¿la ausencia de la variable depende de otra variable observada?
for col in ['edad', 'cargo_mensual']:
    faltante = df_clean[col].isna()
    for grupo in ['tipo_contrato', 'nivel_plan']:
        tabla = pd.crosstab(df_clean[grupo], faltante)
        chi2, p, _, _ = stats.chi2_contingency(tabla)
        tasa = df_clean.groupby(grupo)[col].apply(lambda s: s.isna().mean() * 100).round(1).to_dict()
        veredicto = "depende del grupo -> MAR" if p < 0.05 else "sin relación -> compatible con MCAR"
        print(f"{col:14s} vs {grupo:14s} | p-valor = {p:.4f} | {veredicto}")
        print(f"   % nulos por grupo: {tasa}")

**Lectura de resultados:**
- `edad`: la tasa de nulos es similar entre grupos → compatible con **MCAR**. Es seguro imputar con una medida central.
- `cargo_mensual`: la tasa de nulos es claramente mayor en contratos *Mensual* → **MAR**. La ausencia depende del contrato, pero el valor del cargo depende del plan; imputar con la mediana global sesgaría el valor (un Platino recibiría el cargo de un Básico), por eso se imputa **condicionalmente** con la mediana por `nivel_plan`.
- **MNAR** no es comprobable estadísticamente; en este caso no hay motivo de negocio para sospecharlo.

In [ ]:
# 1. Eliminación por umbral (columnas con >50% de nulos)
umbral_50 = len(df_clean) * 0.5
cols_antes = set(df_clean.columns)
df_clean = df_clean.dropna(axis=1, thresh=umbral_50)
print("Columnas descartadas por umbral:", cols_antes - set(df_clean.columns))

# 2. Imputación condicional por grupo
#    edad (MCAR) -> mediana dentro de cada nivel_plan
df_clean['edad'] = df_clean.groupby('nivel_plan')['edad'].transform(lambda x: x.fillna(x.median()))
#    cargo_mensual (MAR) -> mediana dentro de cada nivel_plan, que es la variable que lo explica
df_clean['cargo_mensual'] = df_clean.groupby('nivel_plan')['cargo_mensual'].transform(lambda x: x.fillna(x.median()))

# 3. Imputación categórica general (moda)
df_clean['tipo_contrato'] = df_clean['tipo_contrato'].fillna(df_clean['tipo_contrato'].mode()[0])

print("\nNulos restantes (fecha_contrato no se usa como predictor):")
print(df_clean.isnull().sum())

> **Nota metodológica:** la guía ubica imputación y winsorización *antes* del split por claridad didáctica. En un proyecto productivo, medianas y percentiles deben calcularse solo con *train* (por ejemplo dentro de un `Pipeline` con `SimpleImputer`); en la **Parte 3** se muestra esa versión.

### Paso 4: Detección y Tratamiento de Outliers
**Objetivo:** cuantificar y mitigar valores atípicos extremos sin eliminar información útil.

| Método | Fórmula / Criterio | Aplicación |
|---|---|---|
| IQR | Límite = Q3 + 1.5 × (Q3 − Q1) | Distribuciones asimétricas (`consumo_datos_gb`) |
| Z-Score | Z = (x − μ) / σ, \|Z\| > 3 | Distribuciones con tendencia gaussiana (`edad`, `cargo_mensual`) |
| Winsorización | `np.clip(x, p1, p99)` | Reemplazo de extremos por percentiles 1 y 99 |

In [ ]:
# --- Detección por IQR en consumo_datos_gb (distribución asimétrica) ---
Q1 = df_clean['consumo_datos_gb'].quantile(0.25)
Q3 = df_clean['consumo_datos_gb'].quantile(0.75)
IQR = Q3 - Q1
lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
out_iqr = ((df_clean['consumo_datos_gb'] < lim_inf) | (df_clean['consumo_datos_gb'] > lim_sup)).sum()
print(f"IQR  | consumo_datos_gb  | límites [{lim_inf:.1f}, {lim_sup:.1f}] | outliers: {out_iqr}")

# --- Detección por Z-Score ---
for col in ['cargo_mensual', 'consumo_datos_gb']:
    z = np.abs(stats.zscore(df_clean[col]))
    print(f"Z>3  | {col:17s} | outliers: {(z > 3).sum()}")

**Comparación:** en `consumo_datos_gb` (asimétrica, con 5 valores de 5000 GB) el Z-Score marca solo esos extremos, porque ellos mismos inflan la desviación estándar (*efecto de enmascaramiento*). El IQR marca muchos más puntos, porque la cola larga de una distribución asimétrica también queda fuera de sus límites aunque sea legítima. Por eso se **winsoriza** (se acotan los extremos) en lugar de eliminar filas. `cargo_mensual` tiene rango acotado y no presenta outliers por Z-Score.

In [ ]:
# --- Tratamiento por Winsorización en consumo_datos_gb (percentiles 1% y 99%) ---
p1 = df_clean['consumo_datos_gb'].quantile(0.01)
p99 = df_clean['consumo_datos_gb'].quantile(0.99)
df_clean['consumo_datos_gb_winsorized'] = np.clip(df_clean['consumo_datos_gb'], p1, p99)
print(f"p1 = {p1:.2f} | p99 = {p99:.2f}")
print(f"Máximo antes: {df_clean['consumo_datos_gb'].max():.0f} | después: {df_clean['consumo_datos_gb_winsorized'].max():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.boxplot(x=df_clean['consumo_datos_gb'], ax=ax[0]); ax[0].set_title('Antes (con outliers)')
sns.boxplot(x=df_clean['consumo_datos_gb_winsorized'], ax=ax[1]); ax[1].set_title('Después (Winsorizado 1%-99%)')
plt.tight_layout(); plt.show()

### Paso 5: Partición Estratificada y Aislamiento de Datos (Train-Test Split)
**Regla de oro:** dividir antes de escalar, codificar o balancear. `stratify=y` mantiene la proporción de la clase minoritaria idéntica en ambos conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean[['edad', 'antiguedad_meses', 'cargo_mensual', 'consumo_datos_gb_winsorized',
              'tipo_contrato', 'nivel_plan']]
y = df_clean['abandono']

# Partición estratificada obligatoria
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño Train: {X_train.shape}, Tamaño Test: {X_test.shape}")
print("\nProporción del target en cada conjunto:")
print(pd.DataFrame({'original': y.value_counts(normalize=True),
                    'train': y_train.value_counts(normalize=True),
                    'test': y_test.value_counts(normalize=True)}).round(4))

### Paso 6: Transformación de Variables e Integración en Pipeline
**Objetivo:** preprocesamiento robusto con `ColumnTransformer`, con `.fit()` **exclusivamente** en *train*.

- **Yeo-Johnson** (`PowerTransformer`): corrige la asimetría de `antiguedad_meses` y `consumo_datos_gb_winsorized`.
- **StandardScaler:** estandariza `edad` y `cargo_mensual` (μ = 0, σ = 1).
- **OneHotEncoder:** `tipo_contrato` (nominal), con `drop='first'` y `handle_unknown='ignore'`.
- **OrdinalEncoder:** `nivel_plan` con orden Básico < Estándar < Premium < Platino.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer

# Definición de columnas
num_cols = ['edad', 'cargo_mensual']
pow_cols = ['antiguedad_meses', 'consumo_datos_gb_winsorized']
cat_nominal = ['tipo_contrato']
cat_ordinal = ['nivel_plan']

orden_plan = [['Básico', 'Estándar', 'Premium', 'Platino']]

# Ensamblado del preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('pow', PowerTransformer(method='yeo-johnson'), pow_cols),
        ('cat_nom', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_nominal),
        ('cat_ord', OrdinalEncoder(categories=orden_plan, handle_unknown='use_encoded_value', unknown_value=-1), cat_ordinal)
    ]
)

# Ajuste EXCLUSIVO sobre Train y transformación de ambos sets
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print("Columnas resultantes:", list(preprocessor.get_feature_names_out()))
print("Forma Train:", X_train_trans.shape, "| Forma Test:", X_test_trans.shape)
print("\nMedia/desvío en Train (deben ser ~0 y ~1):")
print(pd.DataFrame(X_train_trans[:, :4], columns=num_cols + pow_cols).agg(['mean', 'std']).round(3))

### Paso 7: Tratamiento de Desbalanceo con SMOTE, Entrenamiento y Evaluación
**Objetivo:** balancear la clase minoritaria con generación sintética y optimizar la detección de la clase positiva.

**Interpolación de SMOTE:** x_sintético = x_i + λ · (x_vecino − x_i), con λ ~ U(0, 1)

- **CRÍTICO:** SMOTE solo sobre `X_train_trans`. El set de prueba no se toca.
- **Moving Threshold:** bajar el umbral de decisión de 0.50 a 0.25 para ganar sensibilidad (*Recall*).
- Se comparan ambos umbrales y se barre una grilla para elegir con criterio.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_curve,
                             auc, average_precision_score, f1_score, precision_score, recall_score)

# 1. SMOTE exclusivo en Train
print("Train antes de SMOTE:", dict(y_train.value_counts()))
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_trans, y_train)
print("Train después de SMOTE:", dict(pd.Series(y_train_res).value_counts()))

# 2. Entrenamiento de Modelo Cost-Sensitive
clf = RandomForestClassifier(random_state=42, class_weight='balanced')
clf.fit(X_train_res, y_train_res)

# 3. Predicción de probabilidades en Test (que conserva su distribución real)
y_probs = clf.predict_proba(X_test_trans)[:, 1]

# 4. Ajuste de Umbral de Decisión (Moving Threshold a 0.25)
y_pred_custom = (y_probs >= 0.25).astype(int)

# Evaluación de resultados
print("\n--- Matriz de Confusión (Umbral 0.25) ---")
print(confusion_matrix(y_test, y_pred_custom))

print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred_custom))

precision, recall, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)
print(f"Área Bajo la Curva Precision-Recall (PR-AUC): {pr_auc:.4f}")
print(f"Referencia (clasificador al azar) = prevalencia de la clase positiva: {y_test.mean():.4f}")

In [ ]:
# Comparación de umbrales sobre Test
filas = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    pred = (y_probs >= t).astype(int)
    filas.append({'umbral': t,
                  'precision': precision_score(y_test, pred, zero_division=0),
                  'recall': recall_score(y_test, pred),
                  'f1': f1_score(y_test, pred, zero_division=0),
                  'positivos_predichos': pred.sum()})
print(pd.DataFrame(filas).round(3).to_string(index=False))

plt.figure(figsize=(5, 4))
plt.plot(recall, precision, label=f'PR-AUC = {pr_auc:.3f}')
plt.axhline(y_test.mean(), ls='--', c='gray', label='Azar')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Curva Precision-Recall (Test)')
plt.legend(); plt.show()

**Cómo interpretar:** con un 7 % de positivos, la *accuracy* es engañosa (predecir siempre "no abandona" ya daría ~93 %). Por eso se evalúa con **PR-AUC** (comparada contra la prevalencia como línea base) y **F1** de la clase positiva. Bajar el umbral aumenta el *Recall* a costa de *Precision*: en retención de clientes suele convenir, porque contactar a un cliente que no se iba cuesta menos que perder uno que sí.

## 2.3 Lista de Comprobación Final (Checklist Pre-Entrenamiento)
| Verificación | Requisito Técnico | Riesgo si no se cumple |
|---|---|---|
| ✓ | Partición Train/Test realizada previa a transformaciones y balanceo | Fuga de Información (Data Leakage) |
| ✓ | Muestreo estratificado en la división del target | Distorsión en la distribución de la variable objetivo |
| ✓ | `fit()` solo en Train (ColumnTransformer, SMOTE, modelo) | Contaminación estadística de parámetros de evaluación |
| ✓ | Manejo explícito de categorías desconocidas (`handle_unknown`) | Excepciones de código en tiempo de ejecución |
| ✓ | SMOTE ejecutado exclusivamente sobre Train | Métricas de validación falsas e irreales |
| ✓ | Evaluación mediante PR-AUC y F1 en lugar de Accuracy | Paradoja de la Exactitud en datos desbalanceados |
| ✓ | Variables de fuga (`dias_hasta_baja`, `motivo_baja`) eliminadas | Modelo con rendimiento irreal que no funciona en producción |

# Parte 3
## Búsqueda de datasets reales
Selección de datasets públicos para practicar el mismo pipeline sobre datos reales, agrupados por el tipo de desafío. La columna *Pasos del pipeline* indica qué etapas de las Partes 1 y 2 se ejercitan más con cada uno.

> Los tamaños son aproximados. Verificá siempre la versión y la licencia en la página oficial antes de usar un dataset.

### 1. Datos tabulares complejos y feature engineering
Requieren limpieza exhaustiva, varios tipos de datos, alta cardinalidad y creación de variables.

| Dataset | Fuente | Tamaño aprox. | Desafíos | Pasos del pipeline |
|---|---|---|---|---|
| **Online Retail** | [UCI](https://archive.ics.uci.edu/dataset/352/online-retail) / Kaggle | ~540.000 filas, 8 columnas | Cantidades negativas (cancelaciones), `CustomerID` nulo, parseo de fecha/hora, outliers en precio unitario, análisis RFM | 1, 2, 3, 4 |
| **House Prices: Advanced Regression** | [Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) | 1.460 filas × 79 variables | Nulos con significado (sin garaje/piscina), codificación ordinal y nominal, Target Encoding, log/Yeo-Johnson sobre el precio | 3, 4, 6 |
| **Adult / Census Income** | [UCI](https://archive.ics.uci.edu/dataset/2/adult) | ~48.800 filas, 14 variables | Categóricas heterogéneas, espacios y `?` como nulo, categorías de baja frecuencia, escalas distintas; target ~24 % positivo | 2, 3, 6, 7 |

### 2. Desbalanceo severo de clases y detección de anomalías
Ideales para SMOTE/ADASYN, aprendizaje sensible al costo y detección de outliers multivariantes.

| Dataset | Fuente | Tamaño aprox. | Desafíos | Pasos del pipeline |
|---|---|---|---|---|
| **Credit Card Fraud Detection** | [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) | 284.807 filas; 0,172 % fraudes | Ajuste de umbral, SMOTE solo en train, PR-AUC, Isolation Forest / DBSCAN | 5, 6, 7 |
| **IEEE-CIS Fraud Detection** | [Kaggle](https://www.kaggle.com/c/ieee-fraud-detection) | ~590.000 transacciones, +400 columnas (Vesta) | Gran volumen, alineación de timestamps, muchísimos faltantes, variables anónimas correlacionadas | 1, 3, 5, 7 |

### 3. Series temporales y datos secuenciales
Dependencias temporales, autocorrelación e interpolación de vacíos.

| Dataset | Fuente | Tamaño aprox. | Desafíos | Pasos del pipeline |
|---|---|---|---|---|
| **Store Sales – Time Series Forecasting** | [Kaggle](https://www.kaggle.com/competitions/store-sales-time-series-forecasting) | ~3 millones de filas (Corporación Favorita) | Lags, rolling windows, datos exógenos (feriados, promociones, petróleo), calendarios sin *leakage* | 2, 5, 6 |
| **Beijing PM2.5 (Air Quality)** | [UCI](https://archive.ics.uci.edu/dataset/381/beijing-pm2-5-data) | ~43.800 filas horarias | Imputación en secuencias (interpolación, *forward-fill*), resampleo, estacionalidad | 2, 3 |

### 4. Big Data y procesamiento distribuido (PySpark / Dask)
| Dataset | Fuente | Tamaño aprox. | Desafíos | Pasos del pipeline |
|---|---|---|---|---|
| **NYC Yellow Taxi Trip Data** | [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) | Cientos de millones de viajes (archivos mensuales en Parquet) | Memoria, distancia Haversine, coordenadas fuera de la ciudad, procesamiento distribuido | 2, 4 |
| **AWS Open Data / Google Cloud Public Datasets** | [AWS](https://registry.opendata.aws/) / [BigQuery](https://cloud.google.com/bigquery/public-data) | Terabytes (GitHub Activity, NOAA Weather, etc.) | SQL/BigQuery optimizado, particionamiento, ETL escalable | ETL |

### Plataformas y buscadores para descargar datasets
- **[Kaggle Datasets](https://www.kaggle.com/datasets):** filtrar por *Usability* alto o etiquetas como `data-preprocessing`, `time-series`, `imbalanced-data`.
- **[UCI Machine Learning Repository](https://archive.ics.uci.edu/):** filtros por tipo de variable y de tarea.
- **[Google Dataset Search](https://datasetsearch.research.google.com/):** rastrea repositorios académicos, gubernamentales y abiertos.
- **[OpenML](https://www.openml.org/):** datasets estandarizados con flujos de trabajo documentados.

### Criterio de elección para practicar en Colab
Se eligieron dos datasets cargables directamente desde OpenML (sin descargar archivos a mano) que cubren lo más importante de la guía:
- **Adult / Census Income** → limpieza de texto, nulos escritos como `?`, categorías raras, desbalanceo moderado (~24 %).
- **Credit Card Fraud** → desbalanceo extremo (0,172 %), donde SMOTE, umbral y PR-AUC son decisivos.

### 3.1 Pipeline reutilizable (versión sin fuga de datos)
Función que aplica las etapas 2 a 7 sobre **cualquier** dataset tabular con target binario. A diferencia de los pasos de la Parte 2, **la imputación vive dentro del `ColumnTransformer`**, por lo que sus medianas y modas se calculan solo con *train* (sin *leakage*).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, f1_score
from imblearn.over_sampling import SMOTE


def pipeline_completo(df, target, pos_label, drop_cols=(), max_filas=None,
                      umbral=0.25, test_size=0.2, seed=42):
    """Limpieza ligera + split estratificado + preprocesamiento + SMOTE (solo train) + RF + umbral."""
    d = df.drop(columns=list(drop_cols)).copy()

    # Texto: quitar espacios y convertir '?' / 'nan' a NaN real
    for c in d.select_dtypes(include=['object', 'category', 'string']).columns.drop(target, errors='ignore'):
        d[c] = d[c].astype('object').astype(str).str.strip().replace({'?': np.nan, 'nan': np.nan, 'None': np.nan})

    # Target binario: 1 = clase de interés
    y = (d[target].astype(str).str.strip() == str(pos_label)).astype(int)
    X = d.drop(columns=[target])

    # Eliminación por umbral: columnas con >50% de nulos
    X = X.loc[:, X.isna().mean() <= 0.5]

    # Submuestreo estratificado opcional (para datasets muy grandes en Colab)
    if max_filas and len(X) > max_filas:
        X, _, y, _ = train_test_split(X, y, train_size=max_filas, stratify=y, random_state=seed)

    # Partición estratificada ANTES de cualquier transformación
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

    num_cols = X_tr.select_dtypes(include='number').columns.tolist()
    cat_cols = [c for c in X_tr.columns if c not in num_cols]

    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('pow', PowerTransformer(method='yeo-johnson'))]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                min_frequency=0.01, sparse_output=False))]), cat_cols),
    ])
    Xtr_t = pre.fit_transform(X_tr)      # fit solo en train
    Xte_t = pre.transform(X_te)

    Xtr_res, ytr_res = SMOTE(random_state=seed).fit_resample(Xtr_t, y_tr)   # SMOTE solo en train
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=seed)
    clf.fit(Xtr_res, ytr_res)

    probs = clf.predict_proba(Xte_t)[:, 1]
    pred = (probs >= umbral).astype(int)

    print(f"Filas usadas: {len(X)} | variables numéricas: {len(num_cols)} | categóricas: {len(cat_cols)}")
    print(f"Prevalencia de la clase positiva en test: {y_te.mean():.4f}")
    print(f"\n--- Matriz de Confusión (umbral {umbral}) ---\n{confusion_matrix(y_te, pred)}")
    print(f"\n{classification_report(y_te, pred, zero_division=0)}")
    print(f"PR-AUC (average precision): {average_precision_score(y_te, probs):.4f}  (azar = {y_te.mean():.4f})")
    print(f"F1 clase positiva: umbral 0.50 = {f1_score(y_te, (probs >= 0.5).astype(int), zero_division=0):.3f} | "
          f"umbral {umbral} = {f1_score(y_te, pred, zero_division=0):.3f}")
    return clf, pre, (X_te, y_te, probs)

**Prueba rápida de la función** con el dataset de la Parte 2 (ya cargado en `df_raw`), pasándole las dos variables de fuga en `drop_cols`:

In [ ]:
modelo_prueba, pre_prueba, _ = pipeline_completo(
    df_raw.drop_duplicates(),
    target='abandono', pos_label=1,
    drop_cols=['id_cliente', 'fecha_contrato', 'dias_hasta_baja', 'motivo_baja'],
)

### 3.2 Práctica con dataset real 1: Adult / Census Income
`fetch_openml` descarga el dataset desde OpenML. Requiere conexión a internet (Colab la tiene).

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml('adult', version=2, as_frame=True)
df_adult = adult.frame
print(df_adult.shape)
print(df_adult.head())
print(df_adult.isna().sum()[lambda s: s > 0])
print("\nTarget:\n", df_adult['class'].value_counts(normalize=True))

In [ ]:
# 'fnlwgt' es un peso muestral del censo, no una característica de la persona: se excluye.
# 'education' y 'education-num' son redundantes entre sí (misma información); se conserva solo la numérica.
clf_adult, pre_adult, _ = pipeline_completo(
    df_adult, target='class', pos_label='>50K',
    drop_cols=['fnlwgt', 'education'], umbral=0.40
)

### 3.3 Práctica con dataset real 2: Credit Card Fraud (desbalanceo extremo)
Con solo 0,172 % de fraudes, la *accuracy* sería ~99,8 % prediciendo siempre "no fraude": acá se ve por qué importan PR-AUC, SMOTE y el umbral. Se usa un submuestreo estratificado de 60.000 filas para que corra rápido en Colab (la función conserva la proporción de fraudes).

In [ ]:
cc = fetch_openml(data_id=1597, as_frame=True)     # 'creditcard' (Kaggle / ULB) en OpenML
df_cc = cc.frame
print(df_cc.shape)
print(df_cc['Class'].value_counts(normalize=True))

In [ ]:
clf_cc, pre_cc, _ = pipeline_completo(
    df_cc, target='Class', pos_label='1',
    max_filas=60000, umbral=0.25
)

### 3.4 Conclusiones de la práctica con datos reales
- **Adult:** los nulos vienen escritos como `?` y hay espacios sobrantes en el texto; sin la normalización del Paso 2 esas marcas se tratarían como categorías válidas. La clase positiva (>50K) es ~24 %: desbalanceo moderado, donde mover el umbral ya mejora el F1.
- **Credit Card Fraud:** con 0,172 % de positivos la *accuracy* no sirve; la evaluación debe hacerse con PR-AUC (comparada contra la prevalencia) y el umbral se elige según el costo de un fraude no detectado frente a una falsa alarma.
- **En ambos casos** el orden fue el mismo que en las Partes 1 y 2: *split* estratificado primero, imputación y escalado ajustados solo en *train*, SMOTE solo en *train* y evaluación sobre un *test* con la distribución real.